In [ ]:
!pip install accelerate -U
!pip install transformers[torch] -U

In [ ]:
import zipfile


file_zip = 'archive.zip'

with zipfile.ZipFile(file_zip, 'r') as zip_ref:
    zip_ref.extractall('data_source')



In [ ]:
import pandas as pd

path = 'data_source/fakenews-01.2026.csv'
df = pd.read_csv(path)

# Hiển thị 5 dòng đầu tiên
df.head()

,ID,Nhận định (Claim),Phân loại (Verdict),Độ tin cậy (Confidence),Lý do (Reasoning),Tags,Thời gian
0,nvuyujkp3,Tỷ lệ thất nghiệp trong độ tuổi lao động tại V...,Thật,95.0,Số liệu này được công bố chính thức trong báo ...,việc làm hiện nay ở Việt nam;BatchData,2026-02-01T07:45:36.035Z
1,6to24621m,Samsung Thái Nguyên thông báo sẽ sa thải hơn 5...,Tin giả,90.0,Thông tin này không có trong bất kỳ thông cáo ...,việc làm hiện nay ở Việt nam;BatchData,2026-02-01T07:45:36.035Z
2,kikuro23t,Việc làm tại nhà dán nhãn bao bì hoặc chốt đơn...,Tin giả,100.0,Đây là hình thức lừa đảo tuyển dụng phổ biến n...,việc làm hiện nay ở Việt nam;BatchData,2026-02-01T07:45:36.035Z
3,cryy5mu7i,Thủ tướng Chính phủ đặt mục tiêu Việt Nam sẽ đ...,Thật,98.0,Đây là chiến lược quốc gia trọng điểm đã được ...,việc làm hiện nay ở Việt nam;BatchData,2026-02-01T07:45:36.035Z
4,x8pgoqkid,Ngành IT tại Việt Nam hiện đang bão hòa hoàn t...,Gây hiểu lầm,85.0,Thị trường IT chỉ đang thanh lọc các nhân sự c...,việc làm hiện nay ở Việt nam;BatchData,2026-02-01T07:45:36.035Z


In [ ]:
!pip install underthesea

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 82.8 MB/s eta 0:00:00


In [ ]:
import re
from underthesea import word_tokenize

def clean_v_news(text):
    if not isinstance(text, str):
        return ""
    # 1. Chuyển về chữ thường
    text = text.lower()
    # 2. Xóa các ký tự đặc biệt, dấu câu để máy tập trung vào từ vựng
    text = re.sub(r'[^\w\s]', '', text)
    # 3. Tách từ tiếng Việt (Word Segmentation)
    # Kết quả sẽ có dạng: "thủ_tướng chính_phủ" thay vì "thủ tướng chính phủ"
    return word_tokenize(text, format="text")

# Tạo cột mới đã được làm sạch để đối chiếu
df['claim_cleaned'] = df['Nhận định (Claim)'].apply(clean_v_news)

# Xem thử kết quả 5 dòng đầu
df[['Nhận định (Claim)', 'claim_cleaned']].head()

,Nhận định (Claim),claim_cleaned
0,Tỷ lệ thất nghiệp trong độ tuổi lao động tại V...,tỷ_lệ thất_nghiệp trong độ tuổi lao_động tại v...
1,Samsung Thái Nguyên thông báo sẽ sa thải hơn 5...,samsung thái_nguyên thông_báo sẽ sa_thải hơn 5...
2,Việc làm tại nhà dán nhãn bao bì hoặc chốt đơn...,việc_làm tại nhà dán nhãn bao_bì hoặc chốt đơn...
3,Thủ tướng Chính phủ đặt mục tiêu Việt Nam sẽ đ...,thủ_tướng chính_phủ đặt mục_tiêu việt_nam sẽ đ...
4,Ngành IT tại Việt Nam hiện đang bão hòa hoàn t...,ngành it tại việt_nam hiện đang bão_hòa hoàn_t...


In [ ]:
# Kiểm tra xem có những loại nhãn nào
print("Các loại nhãn:", df['Phân loại (Verdict)'].unique())

# Chuyển: Thật -> 0, Tin giả -> 1 (hoặc ngược lại tùy bạn quy định)
# Lưu ý: Nếu có nhãn "Gây hiểu lầm", bạn có thể gộp chung vào "Tin giả"
label_dict = {'Thật': 0, 'Tin giả': 1, 'Gây hiểu lầm': 1}
df['label'] = df['Phân loại (Verdict)'].map(label_dict)

df[['Phân loại (Verdict)', 'label']].head()

Các loại nhãn: ['Thật' 'Tin giả' 'Gây hiểu lầm' 'Tin đồn' 'Chưa xác thực']


,Phân loại (Verdict),label
0,Thật,0.0
1,Tin giả,1.0
2,Tin giả,1.0
3,Thật,0.0
4,Gây hiểu lầm,1.0


In [ ]:
# Lưu lại file đã tiền xử lý hoàn chỉnh

df.to_csv('vnews_ready_to_train.csv', index=False)

In [ ]:
import pandas as pd

# 1. Đọc dữ liệu từ file csv gốc
df = pd.read_csv('vnews_ready_to_train.csv')

# 2. Xử lý khoảng trắng thừa trong tên cột
df.columns = [c.strip() for c in df.columns]

# 3. Khôi phục nhãn (Label) dựa trên cột Phân loại (Verdict)
# Mục đích: Khắc phục lỗi lệch dữ liệu ở cột nhãn gốc
def mapping_label(verdict):
    if str(verdict).strip() == 'Thật':
        return 0
    else:
        return 1

# Thực hiện gán nhãn mới cho toàn bộ tập dữ liệu
df['label'] = df['Phân loại (Verdict)'].apply(mapping_label)

# 4. Loại bỏ các bản ghi khuyết thiếu ở cột nội dung đã tiền xử lý
df_clean = df.dropna(subset=['claim_cleaned']).copy()

# 5. Kiểm tra quy mô dữ liệu sau khi xử lý
print(f"Tổng số dòng dữ liệu hợp lệ: {len(df_clean)}")

if len(df_clean) > 0:
    # Lựa chọn các trường thông tin quan trọng để xuất báo cáo
    target_columns = ['ID', 'Nhận định (Claim)', 'Phân loại (Verdict)', 'claim_cleaned', 'label']
    df_final = df_clean[target_columns]

    # Xuất dữ liệu ra định dạng Excel (.xlsx) để lưu trữ và báo cáo
    output_filename = 'vnews_processed_data.xlsx'
    df_final.to_excel(output_filename, index=False)

    print(f"Kết quả đã được lưu tại file: {output_filename}")
    print("Dữ liệu mẫu sau khi hiệu chỉnh nhãn:")
    print(df_final[['Phân loại (Verdict)', 'label']].head())

Tổng số dòng dữ liệu hợp lệ: 7537
Kết quả đã được lưu tại file: vnews_processed_data.xlsx
Dữ liệu mẫu sau khi hiệu chỉnh nhãn:
  Phân loại (Verdict)  label
0                Thật      0
1             Tin giả      1
2             Tin giả      1
3                Thật      0
4        Gây hiểu lầm      1


In [ ]:
!pip install transformers[torch] accelerate -U
!pip install scikit-learn

In [ ]:
import pandas as pd

# 1. Đọc file Excel
file_name = 'vnews_processed_data.xlsx'
df = pd.read_excel(file_name)

# 2. Xử lý triệt để các dòng trống (NaN)
# Bước này cực kỳ quan trọng để tránh lỗi CUDA khi huấn luyện mô hình
df_clean = df.dropna(subset=['claim_cleaned', 'label']).copy()

# 3. Đảm bảo nhãn (label) là số nguyên
df_clean['label'] = df_clean['label'].astype(int)

# 4. Kiểm tra quy mô dữ liệu sau khi làm sạch
print(f"Số lượng mẫu dữ liệu sẵn sàng huấn luyện: {len(df_clean)}")
print("Cấu trúc các cột:")
print(df_clean.info())

# Hiển thị 5 dòng đầu để kiểm tra nội dung
print("\nDữ liệu mẫu:")
print(df_clean.head())

Số lượng mẫu dữ liệu sẵn sàng huấn luyện: 7537
Cấu trúc các cột:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7537 entries, 0 to 7536
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   ID                   7537 non-null   object
 1   Nhận định (Claim)    7537 non-null   object
 2   Phân loại (Verdict)  7537 non-null   object
 3   claim_cleaned        7537 non-null   object
 4   label                7537 non-null   int64 
dtypes: int64(1), object(4)
memory usage: 294.5+ KB
None

Dữ liệu mẫu:
          ID                                  Nhận định (Claim)  \
0  nvuyujkp3  Tỷ lệ thất nghiệp trong độ tuổi lao động tại V...   
1  6to24621m  Samsung Thái Nguyên thông báo sẽ sa thải hơn 5...   
2  kikuro23t  Việc làm tại nhà dán nhãn bao bì hoặc chốt đơn...   
3  cryy5mu7i  Thủ tướng Chính phủ đặt mục tiêu Việt Nam sẽ đ...   
4  x8pgoqkid  Ngành IT tại Việt Nam hiện đang bão hòa hoàn t...   

  Phân loại 

In [ ]:
from sklearn.model_selection import train_test_split

# --- BƯỚC 1: CHUẨN BỊ DỮ LIỆU ĐẦU VÀO ---
# Lấy nội dung văn bản và nhãn từ bảng dữ liệu đã làm sạch
texts = df_clean['claim_cleaned'].tolist()
labels = df_clean['label'].tolist()

# --- BƯỚC 2: CHIA TẬP DỮ LIỆU (TRAIN/VAL) ---
train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts,
    labels,
    test_size=0.2,
    random_state=42 # Giữ số 42 để kết quả mỗi lần chạy đều giống nhau
)

# --- BƯỚC 3: MÃ HÓA VĂN BẢN (TOKENIZATION) ---
print("Đang tiến hành mã hóa văn bản...")
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=256)
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=256)

print("Mã hóa thành công!")

Đang tiến hành mã hóa văn bản...
Mã hóa thành công!


In [ ]:
import torch
from torch.utils.data import Dataset

# 1. Tạo một Class để đóng gói dữ liệu theo chuẩn PyTorch
class VNewsDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

# 2. Đóng gói train_encodings và val_encodings vào Dataset
train_dataset = VNewsDataset(train_encodings, train_labels)
val_dataset = VNewsDataset(val_encodings, val_labels)

print("Đã khởi tạo train_dataset và val_dataset thành công!")

Đã khởi tạo train_dataset và val_dataset thành công!


In [12]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from transformers import Trainer, TrainingArguments

# 1. Định nghĩa hàm đo lường hiệu suất (Metrics)
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='weighted')
    return {
        'accuracy': acc,
        'f1': f1
    }

# 2. Cấu hình tham số huấn luyện
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    eval_strategy="epoch"
)

# 3. Khởi tạo Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

# 4. Bắt đầu huấn luyện
print("Hệ thống đang khởi động quá trình huấn luyện PhoBERT...")
trainer.train()

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Hệ thống đang khởi động quá trình huấn luyện PhoBERT...


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.428853,0.285481,0.872016,0.871254
2,0.270764,0.283757,0.897878,0.897878
3,0.063024,0.369304,0.908488,0.908487


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1131, training_loss=0.2858892262820237, metrics={'train_runtime': 231.7324, 'train_samples_per_second': 78.051, 'train_steps_per_second': 4.881, 'total_flos': 306725310007380.0, 'train_loss': 0.2858892262820237, 'epoch': 3.0})

In [14]:
# Lưu mô hình và bộ tách từ đã train xong vào thư mục
trainer.save_model("./phobert_vnews_model")
tokenizer.save_pretrained("./phobert_vnews_model")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./phobert_vnews_model/tokenizer_config.json',
 './phobert_vnews_model/vocab.txt',
 './phobert_vnews_model/bpe.codes',
 './phobert_vnews_model/added_tokens.json')

In [15]:
from transformers import pipeline

# Nạp mô hình từ thư mục bạn đã lưu
classifier = pipeline("text-classification", model="phobert_fakenews_model", tokenizer="phobert_fakenews_model")

# Câu test thử
sentence = "Cảnh báo: Loại trái cây này chứa chất gây ung thư cực mạnh, hãy chia sẻ ngay để cứu người!"
result = classifier(sentence)

# Hiển thị kết quả
label = "TIN GIẢ/GÂY HIỂU LẦM" if result[0]['label'] == 'LABEL_1' else "TIN THẬT"
print(f"Dự đoán: {label} (Độ tin cậy: {result[0]['score']*100:.2f}%)")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Dự đoán: TIN THẬT (Độ tin cậy: 57.31%)


In [16]:
import pandas as pd
from collections import Counter

# Lấy các tin được dán nhãn là Tin giả (label = 1)
fake_news = df[df['label'] == 1]['claim_cleaned']

# Tách tất cả các từ và đếm tần suất
all_words = ' '.join(fake_news).split()
word_counts = Counter(all_words)

# Lấy top 20 từ xuất hiện nhiều nhất (bỏ qua các từ chung chung như 'là', 'của'...)
print("Top từ khóa đặc trưng trong tin giả:")
print(word_counts.most_common(20))

Top từ khóa đặc trưng trong tin giả:
[('sẽ', 919), ('các', 847), ('là', 681), ('có_thể', 574), ('của', 550), ('không', 548), ('trong', 545), ('để', 517), ('một', 516), ('và', 475), ('có', 475), ('vào', 468), ('bị', 458), ('hoàn_toàn', 424), ('được', 411), ('tại', 409), ('năm', 386), ('người', 377), ('cho', 337), ('việt_nam', 334)]


In [17]:
import shutil

# Nén thư mục phobert_vnews_model thành file phobert_vnews_model.zip
shutil.make_archive('phobert_vnews_model', 'zip', 'phobert_vnews_model')



'/content/phobert_vnews_model.zip'